# Lab 04: Transformer Block

**Module 04** | Read `notes/04-transformers.pdf` before running this notebook.

Build one transformer encoder block from scratch: multi-head self-attention, feed-forward network, residual connections, and layer normalization.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **self-attention** | Each position in a sequence attends to every other position in the same sequence. |
| **query / key / value** | Three vectors per token: query asks a question, keys are labels, values carry content to mix. |
| **multi-head** | Run several attention operations in parallel, each learning different relationship patterns. |
| **residual connection** | Add the layer input to its output (x + f(x)) so gradients can flow through skip paths. |
| **layer norm** | Normalize activations per token to stabilize training (scale and shift to mean 0, variance 1). |
| **feed-forward** | A small two-layer MLP applied independently to each position after attention. |

Refer back to this table whenever a term appears in code or output.


## What problem are we solving?

Recurrent models process tokens one at a time. **Transformers** let every token look at every other token **in parallel** via **self-attention**. Modern models like BERT and GPT stack many identical encoder layers; here we build **one layer from scratch** so you can see every piece.

## What you will learn

- How **query, key, and value** vectors produce attention weights
- Why we scale dot products and apply **softmax**
- What **multi-head attention** adds over a single head
- How **residual connections** and **layer normalization** keep training stable
- That one encoder layer preserves input shape `(batch, sequence, features)`

No recurrence is used. The sequence length never changes inside one layer.


### Step 1: Scaled dot-product and multi-head attention

**What this code does:** Implements multi-head self-attention, a feed-forward block, and a full transformer encoder layer. Prints the layer structure and parameter count.

**Why we do it:** Reading PyTorch's built-in `TransformerEncoderLayer` hides the internals. Writing it yourself makes each tensor operation visible.


**What to look for in the output**

Printed module tree with MultiHeadSelfAttention, FeedForward, LayerNorm layers, and a parameter count in the tens of thousands.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

BATCH_SIZE = 4
SEQ_LEN = 16
D_MODEL = 64      # Feature dimension per token
NUM_HEADS = 4     # Parallel attention heads
D_FF = 256        # Hidden size inside the feed-forward block
DROPOUT = 0.1


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        # One linear layer produces Q, K, V together
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        b, seq_len, d = x.shape
        qkv = self.qkv(x).chunk(3, dim=-1)
        # Reshape into (batch, num_heads, seq_len, d_head)
        q, k, v = [
            tensor.view(b, seq_len, self.num_heads, self.d_head).transpose(1, 2)
            for tensor in qkv
        ]
        # Scaled dot-product: compatibility between every query-key pair
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = self.dropout(F.softmax(scores, dim=-1))
        mixed = attn @ v  # weighted sum of values
        mixed = mixed.transpose(1, 2).contiguous().view(b, seq_len, d)
        out = self.out(mixed)
        if return_attn:
            return out, attn
        return out


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerEncoderLayerScratch(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Sub-layer 1: self-attention + residual + layer norm
        attn_out = self.self_attn(x)
        x = self.norm1(x + self.dropout(attn_out))
        # Sub-layer 2: feed-forward + residual + layer norm
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x


layer = TransformerEncoderLayerScratch(D_MODEL, NUM_HEADS, D_FF, DROPOUT).to(device)
n_params = sum(p.numel() for p in layer.parameters())
print(layer)
print(f"\nParameters: {n_params:,}")


**Concept: Query, key, value (QKV)**

Think of attention like a dictionary lookup:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I contain?" (labels on each position)
- **Value (V):** "Here is my actual content."

Dot product `Q @ K^T` scores how much each query matches each key. **Softmax** turns scores into weights. The output is a weighted mix of **values**. **Multi-head** runs this several times in parallel so different heads can capture different patterns.


### Step 2: Random input and basic statistics

**What this code does:** Creates a random tensor shaped `[batch, sequence, features]` to stand in for token embeddings, and prints basic statistics.

**Why we do it:** Before tracing shapes through the layer, confirm the input tensor has the expected size and reasonable numeric range.


**What to look for in the output**

Input shape (4, 16, 64). Mean near 0, std near 1 (because of randn), min and max within a few units of zero.


In [ ]:
torch.manual_seed(0)
# Random embeddings: in a real model these come from an Embedding layer or projection
x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)

print(f"Input shape: (batch={BATCH_SIZE}, seq_len={SEQ_LEN}, d_model={D_MODEL})")
print(f"  mean: {x.mean().item():.4f}")
print(f"  std:  {x.std().item():.4f}")
print(f"  min:  {x.min().item():.4f}")
print(f"  max:  {x.max().item():.4f}")


### Step 3: Shape walkthrough through the layer

**What this code does:** Prints tensor shapes after self-attention, after the first residual+norm, after feed-forward, and at the final output.

**Why we do it:** Shape checks build intuition. A single encoder layer maps `(B, T, D)` to `(B, T, D)` without changing sequence length.


**What to look for in the output**

Every intermediate shape should keep batch=4, seq_len=16, d_model=64. Final line: 'Shape check passed'.


In [ ]:
print(f"Input shape:           {tuple(x.shape)}")

with torch.no_grad():
    attn_only = layer.self_attn(x)
    print(f"After self-attention:  {tuple(attn_only.shape)}")

    after_norm1 = layer.norm1(x + layer.dropout(attn_only))
    print(f"After residual+norm1:  {tuple(after_norm1.shape)}")

    ff_out = layer.ff(after_norm1)
    print(f"After feed-forward:    {tuple(ff_out.shape)}")

    y = layer(x)
    print(f"Layer output shape:    {tuple(y.shape)}")

assert y.shape == x.shape, "Encoder layer must preserve (batch, seq, d_model)"
print("\nShape check passed, ready to stack into a full encoder.")


### Step 4: Inspect one attention head's softmax row

**What this code does:** Extracts the attention weights for head 0, query position 0, and prints the full softmax row with a simple bar chart.

**Why we do it:** Each row of the attention matrix is a probability distribution over keys. Row sums must equal 1.0. This is how information routing works inside transformers.


**What to look for in the output**

Weight sum should be 1.000000 (or very close). Each position shows a probability and a '#' bar whose length reflects the weight.


In [ ]:
with torch.no_grad():
    _, attn_weights = layer.self_attn(x, return_attn=True)

head_idx = 0
query_pos = 0
row = attn_weights[0, head_idx, query_pos].cpu()

print(f"Attention weights shape: {tuple(attn_weights.shape)}")
print(f"  (batch, num_heads, seq_len, seq_len)")
print(f"\nHead {head_idx}, query position {query_pos} (attending from token {query_pos}):")
print(f"  sum of weights: {row.sum().item():.6f}  (should be 1.0)")
print(f"  max weight at key position: {row.argmax().item()}  (value {row.max().item():.4f})")
print("  Full softmax row:")
for pos, w in enumerate(row.tolist()):
    bar = "#" * int(w * 40)
    print(f"    pos {pos:2d}: {w:.4f}  {bar}")


**Concept: Residual connections and layer normalization**

A **residual connection** adds the input back to the sub-layer output: `x + f(x)`. This creates a shortcut for gradients during backprop, which helps deep networks train.

**Layer normalization** rescales each token's feature vector to stabilize mean and variance. Together, residuals + layer norm are standard in every transformer block.


### Step 5: Numerical sanity checks

**What this code does:** Verifies the output has reasonable statistics, differs from the input, and contains no NaN values.

**Why we do it:** These checks catch implementation bugs early. A working layer should transform the input without blowing up numerically.


**What to look for in the output**

Output mean/std near reasonable values, mean absolute difference from input > 0, 'Output equals input? False', 'Any NaN? False', and 'All checks passed'.


In [ ]:
with torch.no_grad():
    y = layer(x)
    diff = (y - x).abs().mean().item()

print("Numerical sanity checks:")
print(f"  Output mean: {y.mean().item():.4f}  std: {y.std().item():.4f}")
print(f"  Mean |output - input|: {diff:.4f}")
print(f"  Output equals input? {torch.allclose(y, x)}  (should be False)")
print(f"  Any NaN in output? {torch.isnan(y).any().item()}  (should be False)")
print("\nAll checks passed. One encoder layer is working as expected.")
